In [ ]:
import sys
import subprocess

def ensure_installed(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

ensure_installed(['transformers', 'datasets', 'torch', 'scikit-learn', 'pandas', 'numpy'])


In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    pipeline_device = torch.device('mps')
else:
    device = 'cpu'
    pipeline_device = -1

print({'selected_device': device, 'pipeline_device': str(pipeline_device)})


In [ ]:
dataset = load_dataset('dair-ai/emotion', split='test')
class_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

print({'dataset': 'dair-ai/emotion', 'split': 'test', 'num_rows': len(dataset), 'class_names': class_names})
print(dataset[:3])


In [ ]:
model_name = 'typeform/distilbert-base-uncased-mnli'

classifier = pipeline(
    task='zero-shot-classification',
    model=model_name,
    device=pipeline_device
)

print({'model_name': model_name, 'task': 'zero-shot-classification', 'device': device})


In [ ]:
texts = dataset['text']
true_ids = dataset['label']
true_labels = [class_names[i] for i in true_ids]

outputs = classifier(
    texts,
    candidate_labels=class_names,
    multi_label=False,
    batch_size=16,
    truncation=True
)

pred_labels = [o['labels'][0] for o in outputs]
pred_scores = [float(o['scores'][0]) for o in outputs]
label_to_id = {name: i for i, name in enumerate(class_names)}
pred_ids = [label_to_id[label] for label in pred_labels]

results_df = pd.DataFrame({
    'text': texts,
    'true_label': true_labels,
    'predicted_label': pred_labels,
    'predicted_score': pred_scores
})

print(results_df.head(10).to_dict(orient='records'))


In [ ]:
accuracy = accuracy_score(true_ids, pred_ids)
report = classification_report(true_ids, pred_ids, target_names=class_names, digits=4)

print({
    'model_name': model_name,
    'dataset': 'dair-ai/emotion',
    'split': 'test',
    'num_examples': len(dataset),
    'device': device,
    'accuracy': round(float(accuracy), 6)
})
print(report)


In [ ]:
sample_n = 8
print(results_df.head(sample_n).to_string(index=False))
